![iceberg-logo](https://www.apache.org/logos/res/iceberg/iceberg.png)

### SQL Authoring Note (Kotlin Kernel)

Known limitation (Kotlin kernel): JupyterLab does not apply SQL syntax highlighting inside Kotlin triple-quoted strings (`"""..."""`).

What works in this notebook:
- Use the toolbar **Format SQL** action for query formatting.
- Keep SQL in a dedicated variable and execute with `spark.sql(query)`.

```kotlin
val query = """
SELECT *
FROM nyc.taxis
""".trimIndent()

spark.sql(query).show(100, 0, false)
```


In [ ]:
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder().appName("Jupyter").getOrCreate()

spark

## Load Two Months of NYC Taxi/Limousine Trip Data

For this notebook, we will use the New York City Taxi and Limousine Commision Trip Record Data that's available on the AWS Open Data Registry. This contains data of trips taken by taxis and for-hire vehicles in New York City. We'll save this into an iceberg table called `taxis`.

To be able to rerun the notebook several times, let's drop the table if it exists to start fresh.

In [ ]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS nyc.taxis;
""".trimIndent()).show(100, 0, false)

## First create the table

In [ ]:
spark.sql("""
DROP TABLE IF EXISTS nyc.taxis;
""".trimIndent()).show(100, 0, false)

In [ ]:
spark.sql("""
CREATE TABLE nyc.taxis (
    VendorID              bigint,
    tpep_pickup_datetime  timestamp,
    tpep_dropoff_datetime timestamp,
    passenger_count       double,
    trip_distance         double,
    RatecodeID            double,
    store_and_fwd_flag    string,
    PULocationID          bigint,
    DOLocationID          bigint,
    payment_type          bigint,
    fare_amount           double,
    extra                 double,
    mta_tax               double,
    tip_amount            double,
    tolls_amount          double,
    improvement_surcharge double,
    total_amount          double,
    congestion_surcharge  double,
    airport_fee           double
)
USING iceberg
PARTITIONED BY (days(tpep_pickup_datetime))
""".trimIndent()).show(100, 0, false)

# Write a month of data

In [ ]:
var taxi_df = spark.read().parquet("/home/iceberg/data/yellow_tripdata_2022-01.parquet")
taxi_df.writeTo("nyc.taxis").append()

In [ ]:
spark.sql("""
SELECT *
FROM nyc.taxis
""".trimIndent()).show(100, 0, false)

## Metadata Tables

Iceberg tables contain very rich metadata that can be easily queried. For example, you can retrieve the manifest list for any snapshot, simply by querying the table's `snapshots` table.

In [ ]:
spark.sql("""
SELECT *
FROM nyc.taxis.snapshots
""".trimIndent()).show(100, 0, false)

# Write a month of data

In [ ]:
taxi_df = spark.read().parquet("/home/iceberg/data/yellow_tripdata_2022-02.parquet")
taxi_df.writeTo("nyc.taxis").append()

In [ ]:
spark.sql("""
SELECT *
FROM nyc.taxis.snapshots
ORDER BY committed_at DESC
""".trimIndent()).show(100, 0, false)

## Manifest lists

Now we'll list all the manifests. This is the abovemention `manifest_list` of the current snapshot.

In [ ]:
spark.sql("""
SELECT *
FROM nyc.taxis.manifests
""".trimIndent()).show(100, 0, false)

# Manifests

The next layer is the manifests that has references to the Parquet files.

In [ ]:
spark.sql("""
SELECT *
FROM nyc.taxis.files
""".trimIndent()).show(100, 0, false)

# Flexibility of partitioning

We can easily change the partitioning of the table

In [ ]:
spark.sql("""
SELECT * FROM nyc.taxis.partitions
""".trimIndent()).show(100, 0, false)

In [ ]:
spark.sql("""
ALTER TABLE nyc.taxis DROP PARTITION FIELD days(tpep_pickup_datetime)
""".trimIndent()).show(100, 0, false)

In [ ]:
spark.sql("""
ALTER TABLE nyc.taxis ADD PARTITION FIELD hours(tpep_pickup_datetime)
""".trimIndent()).show(100, 0, false)

In [ ]:
spark.sql("""
SELECT * FROM nyc.taxis.partitions
""".trimIndent()).show(100, 0, false)

In [ ]:
spark.sql("""
CALL system.rewrite_data_files('nyc.taxis')
""".trimIndent()).show(100, 0, false)

In [ ]:
spark.sql("""
SELECT *
FROM nyc.taxis.files
""".trimIndent()).show(100, 0, false)

In [ ]:
spark.sql("""
SELECT *
FROM nyc.taxis.snapshots
ORDER BY committed_at DESC
""".trimIndent()).show(100, 0, false)